# 3D-CNN Exercise Recognition Training

Trains a 3D CNN to recognize exercises from skeleton pose data (YOLO-pose keypoints).
All pipeline logic lives in the `action3d` package (`src/action3d/`); this notebook
orchestrates it. The best model is saved to `output/`.

**Requires:** the private dataset under `data/` (see README).

In [ ]:
from pathlib import Path
import torch

from action3d import (
    GradCAM,
    build_dataloaders,
    evaluate,
    load_all_sequences,
    plot_confusion_matrix,
    plot_gradcam_class_summary,
    plot_training_history,
    save_checkpoint,
    train_model,
    visualize_gradcam_skeleton,
)
from action3d.train import build_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
import numpy as np
np.random.seed(42)

## 1. Configuration

In [ ]:
BASE_DIR = Path('..').resolve()
DATA_DIR = BASE_DIR / 'data'
SKELETON_DIR = DATA_DIR / 'dataset' / 'skeleton' / 'yolo_pose_csv'
LABEL_DIR = DATA_DIR / 'label'
SPLIT_FILE = DATA_DIR / 'split.csv'
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

CONFIG = {
    'sequence_length': 64,
    'stride': 32,
    'num_keypoints': 17,
    'num_coords': 3,
    'batch_size': 32,
    'learning_rate': 1e-3,
    'num_epochs': 50,
    'patience': 10,
    'hidden_channels': [32, 64, 128],
    'dropout': 0.3,
    'test_size': 0.2,
    'val_size': 0.1,
}

## 2. Data Loading

Loads every recording matching `split.csv`, builds sliding-window sequences and a stratified train/validation split.

In [ ]:
data = load_all_sequences(SKELETON_DIR, LABEL_DIR, SPLIT_FILE, CONFIG)

print(f"Train samples: {len(data['X_train'])}")
print(f"Validation samples: {len(data['X_val'])}")
print(f"Test samples: {len(data['X_test'])} (from split.csv)")
print(f"Number of classes: {data['num_classes']}")

## 3. Datasets and DataLoaders

In [ ]:
loaders = build_dataloaders(data, CONFIG)

sample_x, sample_y = next(iter(loaders['train']))
print(f"Sample batch X shape: {sample_x.shape}")  # (batch, 3, T, K)
print(f"Sample batch y shape: {sample_y.shape}")

## 4. Model

In [ ]:
model = build_model(CONFIG, data['num_classes'], device)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Training

Class-weighted cross-entropy, Adam with weight decay, ReduceLROnPlateau, early stopping on validation accuracy.

In [ ]:
best_model_state, history, best_val_acc = train_model(model, loaders, data['y_train'], CONFIG, device)

## 6. Evaluation

In [ ]:
from sklearn.metrics import classification_report

model.load_state_dict(best_model_state)
criterion = torch.nn.CrossEntropyLoss()
test_loss, test_acc, test_preds, test_labels = evaluate(model, loaders['test'], criterion, device)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")

class_names = [f"Exercise {data['idx_to_label'][i]}" for i in range(data['num_classes'])]
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=class_names))

## 7. Training Plots

In [ ]:
plot_training_history(history, output_path=OUTPUT_DIR / 'training_history.png')
plot_confusion_matrix(test_labels, test_preds, class_names, output_path=OUTPUT_DIR / 'confusion_matrix.png')

## 8. Save Model

In [ ]:
save_checkpoint(
    OUTPUT_DIR / 'action_3dcnn_model.pth',
    model_state=best_model_state,
    config=CONFIG,
    num_classes=data['num_classes'],
    label_to_idx=data['label_to_idx'],
    idx_to_label=data['idx_to_label'],
    class_names=class_names,
    best_val_acc=best_val_acc,
    test_acc=test_acc,
    history=history,
)

## 9. Grad-CAM Visualization

Grad-CAM (Gradient-weighted Class Activation Mapping) visualizes which regions of the input
data are most important for the model's prediction. For skeleton-based action recognition,
this shows which temporal segments and keypoints contribute most to each class.

In [ ]:
gradcam = GradCAM(model, target_layer_idx=-1)

test_dataset = loaders['test_dataset']
samples_per_class = {}
for idx, (x, y) in enumerate(test_dataset):
    y_val = y.item()
    if y_val not in samples_per_class:
        samples_per_class[y_val] = (x, y_val, idx)
    if len(samples_per_class) >= data['num_classes']:
        break

print(f"Found samples for {len(samples_per_class)} classes")

In [ ]:
gradcam_results = []

for class_idx, (x, true_class, sample_idx) in samples_per_class.items():
    input_tensor = x.unsqueeze(0).to(device)
    cam, pred_class, pred_prob = gradcam.generate_cam(input_tensor)

    print(
        f"Exercise {data['idx_to_label'][true_class]}: predicted "
        f"{data['idx_to_label'][pred_class]} ({pred_prob:.2%})"
    )

    visualize_gradcam_skeleton(
        x, cam, pred_class, true_class, pred_prob,
        data['idx_to_label'], save_path=OUTPUT_DIR / f'gradcam_class_{data['idx_to_label'][class_idx]}.png',
    )

    gradcam_results.append({
        'true_class': true_class,
        'pred_class': pred_class,
        'pred_prob': pred_prob,
        'correct': pred_class == true_class,
        'cam': cam,
    })

In [ ]:
plot_gradcam_class_summary(gradcam_results, data['idx_to_label'], output_path=OUTPUT_DIR / 'gradcam_class_summary.png')
gradcam.remove_hooks()
print("Grad-CAM visualization complete!")